# ANN Hyperparameter Tuning - full spectrum

**Purpose:** Train and evaluate artificial neural networks (scikit-learn `MLPClassifier`) on the **full 1003-wavenumber spectrum** (no PCA, but z-scored) with nested cross-validation. These results are the 'ANN full' reference lines in Fig. 3 of the manuscript (Section 3.2 and 4.2).

**Three classifiers:** species (binary), benzoate condition (binary, called `metabolites` in the code) and culture age (3 classes).

**Nested CV:**
- *Outer loop:* leave one biological replicate out (3 folds); the held-out replicate is the test set.
- *Inner loop:* `GridSearchCV` (scikit-learn default: 5-fold, accuracy scoring) picks the hyperparameters on the training spectra; the best model is refit on all training spectra of the fold and evaluated on the held-out replicate.

**Scaling:** within each outer fold a `StandardScaler` is fit on the training spectra only (each wavenumber z-scored) and applied to the held-out replicate, so no information from the test replicate enters the scaling. The fitted scaler is not saved to the pickle.

**Grid:** activation (ReLU, logistic, tanh), hidden layer sizes (16 architectures from (100, 50) to (5,)), L2 penalty `alpha` (5 log-spaced values, 1e-5 to 1e-3) and batch size (16, 32, 64, 128). `random_state = 42`, `max_iter = 2000`; solver and learning rate are left at the scikit-learn defaults (Adam, initial learning rate 0.001).

**Output:** `Performance Dictionaries/Performance_ANN_nCV.pkl`, a dictionary `{ "CV 1"|"CV 2"|"CV 3": {"species"|"metabolites"|"age": [X_train, y_train, X_test, y_test, y_predict, GridSearchCV object, best model, confusion matrix, classification report]} }`. It is read by `ML_Performance_Comparison.ipynb`.

In [1]:
# Imports (several are not used in this notebook, e.g. plotly, KernelPCA, SVR, RandomizedSearchCV, 3D plotting).
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items
import numpy as np
import os
import re
import pickle

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits import mplot3d
import matplotlib.ticker as mticker
import matplotlib.transforms as transforms
import matplotlib.patches as patches
import matplotlib.cm as cm

import plotly.express as px
import scipy

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, KernelPCA

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Load the preprocessed dataset produced by Data_Preprocessing_Pipeline.ipynb
filename = "Preprocessed_Raman_Spectra_2025.pkl"
with open(filename, "rb") as f:
    loaded_data = pickle.load(f)
    
print(loaded_data['Spectra'].shape)

(360, 1003)


In [3]:
# Show the preprocessing settings stored with the data
loaded_data["Preprocess conditions"]

{'order': 'SI --> CS --> BSR --> SG --> ED',
 'CS': [1, 0.01],
 'BSR': [100, 1],
 'SG': [15, 2],
 'ED': [400, 1800]}

In [4]:
# Unpack the dictionary into separate arrays (spectra, wavenumbers, labels, replicate IDs)
wavenumber = loaded_data["wavenumber"]
spectral = loaded_data["Spectra"]

bacteria_species = loaded_data["species label"]
culture_age = loaded_data['culture age']
cul_cond = loaded_data["culture condition"]

biological_rep = loaded_data["biological replicate"]
technical_rep = loaded_data["technical replicate"]

In [5]:
# Translate the condition codes into readable labels: A = 0 mM benzoate, C = 20 mM benzoate
condition_dict = {"A" : "0 mM benzoate", "C" : "20 mM benzoate"}
culture_condition = []
for i in range(len(cul_cond)):
    culture_condition.append(condition_dict[cul_cond[i]])
culture_condition = np.array(culture_condition)

# Biological Replicate Seperation

In [6]:
# Build the label matrix y_data_full (one row per spectrum, three columns).
# Labels are integer-encoded in sorted order of the unique values:
#   col 0 = species:   CBG (C. glutamicum) = 0, PP (P. putida) = 1
#   col 1 = condition: 0 mM benzoate = 0, 20 mM benzoate = 1  (called 'metabolites' in the code below)
#   col 2 = culture age: 5 h = 0, 24 h = 1, 48 h = 2
# X_data_full is the matrix of preprocessed spectra (360 x 1003).
label_species = np.unique(bacteria_species)
label_condition = np.unique(culture_condition)
label_age = np.unique(culture_age)

species_dict = {}
for i in range(len(label_species)):
    species_dict[label_species[i]] = i
    
condition_dict = {}
for i in range(len(label_condition)):
    condition_dict[label_condition[i]] = i
    
age_dict = {}
for i in range(len(label_age)):
    age_dict[label_age[i]] = i
    
y_output = []
for i in range(len(bacteria_species)):
    temp = [species_dict[bacteria_species[i]],
           condition_dict[culture_condition[i]],
            age_dict[culture_age[i]]
           ]
    y_output.append(np.array(temp))

y_data_full = np.array(y_output)
X_data_full = spectral

print(X_data_full.shape, y_data_full.shape)

(360, 1003) (360, 3)


In [7]:
# Evaluate one outer fold: for each of the three classification tasks tune an ANN by grid search on the training spectra, then predict the held-out spectra.
# Inputs are the z-scored full spectra (the scaling is done in the outer-fold loop below; no PCA in this full-spectrum version).
# Returns {task: [X_train, y_train, X_test, y_test, y_predict, GridSearchCV object, best model, confusion matrix, classification report]}.
def nCV_eval(train_X, test_X, train_y, test_y):
    # Names of the three classification tasks (column j of the label matrix). 'metabolites' = benzoate condition (0 vs 20 mM).
    names = ["species", "metabolites", "age"]
    overall_performance = {}
    # rs is defined but not used; the models below set random_state = 42 directly
    rs = 42

    for j in range(len(names)):

        # Training / test inputs: all 1003 wavenumbers; labels: column j of the label matrix
        X_data_train = train_X
        y_data_train = train_y[:,j]

        X_data_test = test_X
        y_data_test = test_y[:,j]

        print(names[j], X_data_train.shape, y_data_train.shape)
        
        # Hyperparameter grid (full-spectrum version): 3 activations x 16 architectures x 5 alphas x 4 batch sizes
        # activation = hidden layer activation; hidden_layer_sizes = neurons per hidden layer, e.g. (100, 50) = two hidden layers;
        # alpha = L2 regularisation strength; batch_size = mini-batch size for the Adam optimiser
        param_grid = {"activation": ["relu", "logistic", "tanh"],
                      "hidden_layer_sizes" : [(100, 50), (100, 20),
                                              (100, 10), (100, 5),
                                              (50, 20), (50, 10), (50, 5), 
                                              (20, 10), (20, 5), 
                                              (10, 10), (10, 5), 
                                              (100,), (50,), (20,), (10,), (5,)],
                      "alpha" : np.logspace(-5, -3, 5),
                      "batch_size" : [16, 32, 64, 128]}

        # MLPClassifier with a fixed seed and up to 2000 training iterations; other settings are scikit-learn defaults
        # (solver 'adam', learning_rate_init 0.001, log-loss)
        ANN_model = MLPClassifier(random_state = 42, 
                                  max_iter = 2000)

        # Inner cross-validation: GridSearchCV (default 5-fold CV, scored by accuracy); n_jobs = -1 uses all CPU cores.
        # refit = True (default), so the best parameter set is refit on all training spectra of this outer fold.
        ANN_grid = GridSearchCV(ANN_model, 
                                 param_grid, 
                                 n_jobs = -1, 
                                 verbose = 0, 
                                 return_train_score = True)

        ANN_grid.fit(X_data_train, y_data_train)

        # Best model found by the grid search, evaluated on the held-out biological replicate
        ANN_best_model = ANN_grid.best_estimator_
        y_predict = ANN_best_model.predict(X_data_test)

        # Test-set classification report (precision/recall/F1/accuracy) and confusion matrix
        report = classification_report(y_data_test, y_predict, output_dict = True)
        con_mat = confusion_matrix(y_data_test, y_predict)

        # Store inputs, predictions, grid-search object, best model and metrics for later analysis
        overall_performance[names[j]] = [X_data_train, y_data_train, X_data_test, y_data_test, 
                                         y_predict, ANN_grid, ANN_best_model, con_mat, report]

    return overall_performance

In [8]:
# Outer loop = leave-one-biological-replicate-out (3-fold CV). Each fold trains on two biological replicates and tests on the held-out one,
# so spectra from the same culture never appear in both training and test sets (manuscript Section 3.2).
# Results for each fold are collected in NCV_performance['CV <replicate>'].
import time

# Biological replicate IDs (1, 2, 3); each takes a turn as the test set
uni_br = np.unique(biological_rep)
NCV_performance = {}

for i in range(len(uni_br)):
    strcv = "CV " + str(uni_br[i])
    
    print(strcv)
    
    # Boolean masks: spectra from the other replicates (train) and from the held-out replicate (test)
    ind_train = biological_rep != uni_br[i]
    ind_test = biological_rep == uni_br[i]
    
    train_X_full_us = X_data_full[ind_train,:]
    test_X_full_us = X_data_full[ind_test,:]
    
    # Z-score the spectra: the scaler is fit on the TRAINING spectra of this outer fold only (zero mean, unit variance per wavenumber)
    # and then applied unchanged to the held-out replicate, so nothing from the test replicate leaks into the scaling. (train_X_full_us / test_X_full_us = unscaled)
    SS_overall = StandardScaler()
    
    train_X_full = SS_overall.fit_transform(train_X_full_us)
    test_X_full = SS_overall.transform(test_X_full_us)
    
    train_y_data = y_data_full[ind_train,:]
    test_y_data = y_data_full[ind_test,:]
    
    # Tune and evaluate the three ANNs for this fold on the scaled spectra
    temp = nCV_eval(train_X_full,
                    test_X_full, 
                    train_y_data, 
                    test_y_data)


    # Print the test accuracy of each classifier for this fold
    for n in list(temp.keys()):
        print(n + " accuracy: ", round(temp[n][-1]["accuracy"], 2))

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
        
    NCV_performance[strcv] = temp
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

CV 1
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.93
metabolites accuracy:  0.78
age accuracy:  0.82
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 2
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.93
metabolites accuracy:  0.83
age accuracy:  0.87
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 3
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.91
metabolites accuracy:  0.82
age accuracy:  0.84
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [9]:
# Save the nested-CV results (input for ML_Performance_Comparison.ipynb)
import pickle

my_dictionary = NCV_performance
 
# Open a file in write binary mode ('wb')
with open('Performance Dictionaries\\Performance_ANN_nCV.pkl', 'wb') as file:
    # Serialize the dictionary and write it to the file
    pickle.dump(my_dictionary, file)